# データサイエンス実践A 第15講：時系列・自分で選んだデータでグラフと回帰
### 新潟大学 創生学部（2026年度）

1. 日付の列を**日付の型**にして、**時系列**（時間の流れに沿ったデータ）のグラフを描く。
1. **第３回課題**：自分で選んだデータでグラフと単回帰。ひな形の使い方を確かめる。
1. セルは**上から順番に**実行すること（`Shift + Enter` で実行して次のセルへ進む）。

## 目次
**[前回の振り返り](#前回の振り返り)**

**[AQ. 時系列](#AQ.-時系列)**
- [AQ-0 今日の完成形](#AQ-0-今日の完成形)
- [AQ-1 日付の列を日付の型にする](#AQ-1-日付の列を日付の型にする)
- [AQ-2 時系列のグラフを描く](#AQ-2-時系列のグラフを描く)
- [AQ-3 期間を絞る](#AQ-3-期間を絞る)

**[AR. 自分で選んだデータでグラフと回帰](#AR.-自分で選んだデータでグラフと回帰)**
- [AR-1 第３回課題のひな形](#AR-1-第３回課題のひな形)
- [AR-2 提出前のチェックリスト](#AR-2-提出前のチェックリスト)

**[第３回課題](#第３回課題)**

**[AS. 発展（できる人だけ）](#AS.-発展（できる人だけ）)**
- [AS-1 移動平均](#AS-1-移動平均)
- [AS-2 重回帰と標準化](#AS-2-重回帰と標準化)

**[まとめ](#まとめ)**

---
# 前回の振り返り

- 前回は**回帰**：`res = ols('y の列名 ~ x の列名', data=df).fit()` で、散布図にいちばん合う直線を求めた
- 傾きは「x が 1 増えると y がどれだけ増えるか」、決定係数は直線の合い具合（0〜1）
- 第13講で、e-Stat などのデータを **Excel で整えて**読む手順を確かめた。今日はそのデータで第３回課題に取り組む

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

from statsmodels.formula.api import ols

---
# AQ. 時系列

## AQ-0 今日の完成形

ダウ平均株価（アメリカの代表的な株価の指標）の、2000年以降の週ごとの終値。リーマンショック（2008年）とコロナショック（2020年）に線を引いている。

1. **全部コピーして**自分の notebook のセルに貼り付け、実行する
1. まずは**期間や線の位置を変えて**、グラフのどこが変わるかを確かめる
1. そのあと部品（[1]〜[3]）の働きを1つずつ学ぶ（AQ-1〜AQ-3）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

df_dji = pd.read_excel('dsa2026data.xlsx', sheet_name='dji')
df_dji['Date'] = pd.to_datetime(df_dji['Date'])  # [1] 日付の型にする

plt.figure(figsize=(10, 5))
plt.plot(df_dji['Date'], df_dji['Close'], color='k', linewidth=1)  # [2] 時系列
plt.axvline(x=pd.Timestamp('2008-09-15'), color='r', linestyle='--', label='リーマンショック')  # [3] 日付の位置
plt.axvline(x=pd.Timestamp('2020-03-01'), color='b', linestyle='--', label='コロナショック')  # [3] 日付の位置
plt.xlim(pd.Timestamp('2000-01-01'), df_dji['Date'].max())  # [3] 期間
plt.ylim(0, 50000)
plt.title('ダウ平均株価（2000年〜、週ごとの終値）', fontsize=16)
plt.ylabel('USD', fontsize=14)
plt.legend()
plt.show()

### 変えてみよう（1か所変えたら、もう一度実行する）

- `'2000-01-01'` → `'2018-01-01'`（期間を短くする。`plt.ylim` も `(15000, 50000)` などに）
- `'Close'` → `'High'`（週の最高値）
- `axvline` の日付を、自分の誕生日や入学した日に変える

| 番号 | 部品 | 書き方 | 学ぶところ |
|---|---|---|---|
| [1] | 日付の型にする | `df['Date'] = pd.to_datetime(df['Date'])` | AQ-1 |
| [2] | 時系列のグラフ | `plt.plot(df['Date'], df['Close'])` | AQ-2 |
| [3] | 日付で位置・期間を指定 | `pd.Timestamp('2008-09-15')` | AQ-3 |

In [ ]:
# 上のコードを貼り付けて、値を変えてみよう

## AQ-1 日付の列を日付の型にする

- `dji` シートは、1896年から2025年までの週ごとのダウ平均株価（6729行）
- 列：`Date`（日付）、`Open`（始値）、`High`（高値）、`Low`（安値）、`Close`（終値）、`Volume`（出来高）

In [ ]:
df_dji = pd.read_excel('dsa2026data.xlsx', sheet_name='dji')
print(df_dji.dtypes)
df_dji.head()

- `Date` の型が `object`（文字）になっている。このままだと、横軸に日付が**文字として**並ぶ（間隔や順番が正しくならない）
- `pd.to_datetime(df['列名'])` で**日付の型**（`datetime64`）にする

In [ ]:
df_dji['Date'] = pd.to_datetime(df_dji['Date'])
print(df_dji.dtypes)

- `Date` が `datetime64` になった。これで横軸が年月日として正しく並ぶ
- Excel で日付に見えていても、読み込むと文字になることがある。**時系列では、最初に `to_datetime` しておく**と覚える

## AQ-2 時系列のグラフを描く

- 横軸を日付の列にして、折れ線グラフ（第３講）で描く。点が多いので `marker` は付けない

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_dji['Date'], df_dji['Close'], color='k', linewidth=1)
plt.ylabel('USD')
plt.show()

- 1990年代より前は、値が小さすぎてほとんど0に見える
- `plt.yscale('log')`（対数目盛、第８講）にすると、10倍ごとに同じ間隔になり、昔の動きも見える

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_dji['Date'], df_dji['Close'], color='k', linewidth=1)
plt.yscale('log')
plt.ylabel('USD（対数目盛）')
plt.show()

- 1929年の世界恐慌の大きな下落が見える

## AQ-3 期間を絞る

- 日付は `pd.Timestamp('年-月-日')` で書く。`plt.xlim` や `plt.axvline` の位置に使える
- グラフの表示範囲だけでなく、**データそのものを絞る**こともできる：`df[df['Date'] >= '2020-01-01']`（条件に合う行だけ）

In [ ]:
df_recent = df_dji[df_dji['Date'] >= '2020-01-01']
print(len(df_recent), '週')

plt.figure(figsize=(10, 4))
plt.plot(df_recent['Date'], df_recent['Close'], marker='.', linewidth=1)
plt.axvline(x=pd.Timestamp('2020-03-01'), color='b', linestyle='--')
plt.ylabel('USD')
plt.show()

- `df[条件]` は第10講の `df_tip[df_tip['day'] == 'Sat']` と同じ書き方
- 年だけの列（`accident` シートの `年` など）は数字のままでよい。`to_datetime` が要るのは「2020-03-01」のような**年月日**の列

---
# AR. 自分で選んだデータでグラフと回帰

## AR-1 第３回課題のひな形

1. 興味のあるデータを e-Stat などから探す（**2つの量の関係**を見られるもの）
1. 第13講の手順で Excel を整え、notebook と同じ `dsa` フォルダ（いつもの場所）に置く
1. 下のセルの `ファイル名`・`シート名`・`列名`（`x_name`、`y_name`）を書き換える
1. 散布図と回帰直線を見て、**傾き**と**決定係数**を言葉で説明する

- 下は `accident` シート（交通事故件数と負傷者数、2000〜2023年）で書いた例

In [ ]:
df = pd.read_excel('dsa2026data.xlsx', sheet_name='accident')
x_name = '交通事故件数'
y_name = '負傷者数'

res = ols(f'{y_name} ~ {x_name}', data=df).fit()
print(res.params)
print('決定係数', res.rsquared)

plt.figure(figsize=(8, 5))
plt.scatter(df[x_name], df[y_name], zorder=2)
plt.plot(df[x_name], res.predict(), color='r', zorder=3)
plt.title(f'{y_name} と {x_name}（決定係数 {res.rsquared:.2f}）')
plt.xlabel(x_name + '（件）')
plt.ylabel(y_name + '（人）')
plt.show()

- `f'{y_name} ~ {x_name}'` は f 文字列（第14講）。`'負傷者数 ~ 交通事故件数'` と書いたのと同じ
- 書き換えるのは、上の3行（`sheet_name`、`x_name`、`y_name`）と、軸の単位だけ
- 傾き 約1.26：「事故が1件増えると、負傷者は約1.3人増える」。決定係数は 1 に近い（ほぼ直線に並ぶ）

### 年ごとの変化も合わせて見る
- x を `'年'` にすると、「1年でどれだけ増えた（減った）か」が傾きになる。時系列の傾向を見るときに使う

In [ ]:
res_year = ols('負傷者数 ~ 年', data=df).fit()
print(res_year.params)

plt.scatter(df['年'], df['負傷者数'], zorder=2)
plt.plot(df['年'], res_year.predict(), color='r', zorder=3)
plt.xlabel('年')
plt.ylabel('負傷者数（人）')
plt.show()

- 傾きがマイナス：負傷者数は1年に約4万人ずつ減っている。ただし点は直線より曲がって並んでいる（AO-8 の「まず散布図を見る」）

### 自分のデータで書いてみよう

In [ ]:
# df = pd.read_excel('ファイル名.xlsx', sheet_name='シート名')
# x_name = '列名'
# y_name = '列名'
# 上の AR-1 のセルの続きを写して、書き換えよう

## AR-2 提出前のチェックリスト

第12講の「伝わるグラフ」のチェックリストに、回帰の分を足したもの。

| チェック | どうする |
|---|---|
| データの出どころを書いた | 「出典：e-Stat ○○統計（○年）」など |
| 軸の名前・単位がある | `plt.xlabel`・`plt.ylabel` に単位まで |
| 棒グラフの縦軸が 0 から | `plt.ylim(0, …)` |
| 文字が読める大きさ | `fontsize=14` など |
| 傾きを言葉で説明した | 「x が 1 増えると、y は○○増える」 |
| 決定係数を書き、合い具合を説明した | 「0.8 なので、よく合っている」 |
| 原因と言いすぎていない | 「一緒に動く」と「原因」は違う（第14講 AO-8） |

---
# 第３回課題

> **第３回課題（11/27 出題・12/3 締切）**：自分で選んだデータで、グラフと単回帰。重回帰は任意。詳しくは授業HPを見ること。

- データ探し → Excel で整える（第13講）→ グラフ（第３〜12講）→ 回帰（第14講・今日の AR-1）の順に進める
- AR-2 のチェックリストで、提出前に確かめる

---
# AS. 発展（できる人だけ）

## AS-1 移動平均

- 時系列は細かく上下するので、**移動平均**（直近の何個かの平均）でならすと傾向が見やすい
- `df['列名'].rolling(52).mean()`：直近52週（約1年）の平均

In [ ]:
df_recent = df_dji[df_dji['Date'] >= '2010-01-01']
ma = df_recent['Close'].rolling(52).mean()

plt.figure(figsize=(10, 4))
plt.plot(df_recent['Date'], df_recent['Close'], color='lightgray', label='週ごと')
plt.plot(df_recent['Date'], ma, color='r', label='52週移動平均')
plt.ylabel('USD')
plt.legend()
plt.show()

- 最初の51週は平均が計算できないので、線が始まらない

## AS-2 重回帰と標準化

- x を2つ以上使う：`'tip ~ total_bill + size'`（食事代と人数でチップを説明する）

In [ ]:
df_tip = pd.read_excel('dsa2026data.xlsx', sheet_name='tip_data')
res2 = ols('tip ~ total_bill + size', data=df_tip).fit()
print(res2.params)
print('決定係数', res2.rsquared)

- `total_bill` の係数：人数が同じなら、食事代が1ドル増えるとチップが約0.09ドル増える
- `size` の係数：食事代が同じなら、1人増えるとチップが約0.19ドル増える
- 「食事代と人数、どちらが効いているか」は、単位（ドルと人）が違うのでこのままでは比べられない
- 各列を**標準化**（平均0・標準偏差1）してから回帰すると、係数の大きさで比べられる

In [ ]:
cols = ['tip', 'total_bill', 'size']
df_z = (df_tip[cols] - df_tip[cols].mean()) / df_tip[cols].std()

res3 = ols('tip ~ total_bill + size', data=df_z).fit()
print(res3.params)
print('決定係数', res3.rsquared)

- 標準化した係数は `total_bill` の方が大きい → チップには食事代の方が強く効いている
- 決定係数は標準化しても変わらない
- 切片の `2.775558e-16` は「2.78 × 10 の −16 乗」＝ほぼ 0（標準化すると切片は 0 になる）

---
# まとめ

| やりたいこと | 書き方 |
|---|---|
| 日付の型にする | `df['Date'] = pd.to_datetime(df['Date'])` |
| 時系列のグラフ | `plt.plot(df['Date'], df['列名'])` |
| 日付の位置・期間 | `pd.Timestamp('2020-03-01')`、`plt.xlim(開始, 終わり)` |
| 期間でデータを絞る | `df[df['Date'] >= '2020-01-01']` |
| 自分のデータで回帰 | AR-1 のひな形の `sheet_name`・`x_name`・`y_name` を書き換える |

> **第３回課題**：12/3 締切。AR-2 のチェックリストで確かめてから提出する。

15回おつかれさまでした。ここまでの notebook は、グラフを描くときの**ひな形集**として使える。困ったら第11講の「グラフの型の一覧」から探す。